In [2]:
import pandas as pd
import os
import glob

# Define expected patterns and find actual files
def find_file(pattern):
    files = glob.glob(pattern)
    if files:
        return files[0]
    return None

# 1. Identify Files
file_thesis_A = find_file('*ThesisData_A_OH_plus_others_MDS_Targets*.csv')
file_thesis_B = find_file('*ThesisData_B_FP_plus_others_MDS_Targets*.csv')

file_labels_A = find_file('*cluster_labels_matrix_samples_A_OH_plus_others*.csv')
file_labels_B = find_file('*cluster_labels_matrix_samples_B_FP_plus_others*.csv')

# Detail files (Uploaded in turn 27)
file_detail_OH_k50 = 'Detail_OH_Clusters_linear_top3_k50.csv'
file_detail_OH_k10 = 'Detail_OH_Clusters_linear_top3_k10.csv'
file_detail_FP_k50 = 'Detail_FP_Clusters_linear_top3_k50.csv'
file_detail_FP_k10 = 'Detail_FP_Clusters_linear_top3_k10.csv'

print(f"Found Thesis A: {file_thesis_A}")
print(f"Found Thesis B: {file_thesis_B}")
print(f"Found Labels A: {file_labels_A}")
print(f"Found Labels B: {file_labels_B}")

# Output files
OUT_TABLE_MAIN  = 'Table_4_x_Sweet_Spot_Summary.csv'
OUT_TABLE_OH_ALL = 'Table_Appendix_OH_All_Clusters_k50.csv'
OUT_TABLE_FP_ALL = 'Table_Appendix_FP_All_Clusters_k50.csv'

# Helper to load and merge
def load_and_merge(f_thesis, f_labels):
    if not f_thesis or not f_labels or not os.path.exists(f_thesis) or not os.path.exists(f_labels):
        print(f"[WARN] Missing files for merge: {f_thesis}, {f_labels}")
        return None
    
    try:
        df_t = pd.read_csv(f_thesis)
        if 'Unnamed: 0' in df_t.columns:
            df_t.rename(columns={'Unnamed: 0': 'ID'}, inplace=True)
        
        df_l = pd.read_csv(f_labels)
        # Merge on ID
        df = pd.merge(df_t, df_l, on='ID', how='inner')
        return df
    except Exception as e:
        print(f"[ERROR] Merge failed: {e}")
        return None

def load_detail(path):
    if not os.path.exists(path):
        print(f"[WARN] Detail file not found: {path}")
        return None
    return pd.read_csv(path)

# Helper to get stats
def get_stats(df, col_k, cluster_id, detail_df, type_name):
    subset = df[df[col_k] == cluster_id]
    if subset.empty:
        return {'Count': 0, 'Mean_PCE': 0, 'Std_Dev': 0, 'Max_PCE': 0, 'Major_Component': "N/A"}
        
    pce_data = subset['PCEmax']
    
    desc = "N/A"
    if detail_df is not None:
        row = detail_df[detail_df['Cluster_ID'] == cluster_id]
        if not row.empty:
            col_comp = 'Major_Material' if type_name == 'OH' else 'Major_Fragment'
            raw_name = str(row[col_comp].values[0])
            desc = raw_name.replace("SMILESsname", "").replace("p1M_", "").replace("nM_", "").replace("inM_", "")
    
    return {
        'Count': len(subset),
        'Mean_PCE': pce_data.mean(),
        'Std_Dev': pce_data.std(),
        'Max_PCE': pce_data.max(),
        'Major_Component': desc
    }

# Main Process
df_A = load_and_merge(file_thesis_A, file_labels_A)
df_B = load_and_merge(file_thesis_B, file_labels_B)

det_OH_k50 = load_detail(file_detail_OH_k50)
det_OH_k10 = load_detail(file_detail_OH_k10)
det_FP_k50 = load_detail(file_detail_FP_k50)
det_FP_k10 = load_detail(file_detail_FP_k10)

# --- Generate Table 4.x ---
if df_A is not None and df_B is not None:
    print("\nGenerating Table 4.x...")
    rows = []
    
    # 1. OH k=10 Best
    best_id = df_A.groupby('linear_top3_k10')['PCEmax'].mean().idxmax()
    st = get_stats(df_A, 'linear_top3_k10', best_id, det_OH_k10, 'OH')
    rows.append(['OH (Material)', 10, best_id, st['Count'], st['Mean_PCE'], st['Std_Dev'], st['Max_PCE'], st['Major_Component']])
    
    # 2. OH k=50 Best (Cluster 19)
    best_id_50 = 19
    st = get_stats(df_A, 'linear_top3_k50', best_id_50, det_OH_k50, 'OH')
    rows.append(['OH (Material)', 50, best_id_50, st['Count'], st['Mean_PCE'], st['Std_Dev'], st['Max_PCE'], st['Major_Component']])
    
    # 3. FP k=10 Best
    best_id = df_B.groupby('linear_top3_k10')['PCEmax'].mean().idxmax()
    st = get_stats(df_B, 'linear_top3_k10', best_id, det_FP_k10, 'FP')
    rows.append(['FP (Structure)', 10, best_id, st['Count'], st['Mean_PCE'], st['Std_Dev'], st['Max_PCE'], st['Major_Component']])
    
    # 4. FP k=50 Best (Cluster 18)
    best_id_50_fp = 18
    st = get_stats(df_B, 'linear_top3_k50', best_id_50_fp, det_FP_k50, 'FP')
    rows.append(['FP (Structure)', 50, best_id_50_fp, st['Count'], st['Mean_PCE'], st['Std_Dev'], st['Max_PCE'], st['Major_Component']])
    
    df_table4x = pd.DataFrame(rows, columns=['Dataset', 'Resolution (k)', 'Cluster ID', 'N', 'Mean PCE (%)', 'Std Dev', 'Max PCE (%)', 'Major Component'])
    df_table4x.to_csv(OUT_TABLE_MAIN, index=False)
    print(df_table4x)
    
    # --- Generate Appendix Tables (Full Ranking) ---
    
    # OH Ranking
    if det_OH_k50 is not None:
        cids = sorted(df_A['linear_top3_k50'].unique())
        rr = []
        for cid in cids:
            st = get_stats(df_A, 'linear_top3_k50', cid, det_OH_k50, 'OH')
            rr.append([cid, st['Count'], st['Mean_PCE'], st['Std_Dev'], st['Max_PCE'], st['Major_Component']])
        df_r_oh = pd.DataFrame(rr, columns=['Cluster ID', 'N', 'Mean PCE (%)', 'Std Dev', 'Max PCE (%)', 'Major Component'])
        # Fix sort column name
        df_r_oh = df_r_oh.sort_values('Mean PCE (%)', ascending=False)
        df_r_oh.to_csv(OUT_TABLE_OH_ALL, index=False)
        print(f"\nSaved OH Ranking to {OUT_TABLE_OH_ALL}")
        
    # FP Ranking
    if det_FP_k50 is not None:
        cids = sorted(df_B['linear_top3_k50'].unique())
        rr = []
        for cid in cids:
            st = get_stats(df_B, 'linear_top3_k50', cid, det_FP_k50, 'FP')
            rr.append([cid, st['Count'], st['Mean_PCE'], st['Std_Dev'], st['Max_PCE'], st['Major_Component']])
        df_r_fp = pd.DataFrame(rr, columns=['Cluster ID', 'N', 'Mean PCE (%)', 'Std Dev', 'Max PCE (%)', 'Major Component'])
        # Fix sort column name
        df_r_fp = df_r_fp.sort_values('Mean PCE (%)', ascending=False)
        df_r_fp.to_csv(OUT_TABLE_FP_ALL, index=False)
        print(f"Saved FP Ranking to {OUT_TABLE_FP_ALL}")

else:
    print("[ERROR] Could not load main dataframes.")

Found Thesis A: None
Found Thesis B: None
Found Labels A: None
Found Labels B: None
[WARN] Missing files for merge: None, None
[WARN] Missing files for merge: None, None
[WARN] Detail file not found: Detail_OH_Clusters_linear_top3_k50.csv
[WARN] Detail file not found: Detail_OH_Clusters_linear_top3_k10.csv
[WARN] Detail file not found: Detail_FP_Clusters_linear_top3_k50.csv
[WARN] Detail file not found: Detail_FP_Clusters_linear_top3_k10.csv
[ERROR] Could not load main dataframes.


In [3]:
import pandas as pd
import os
import glob

# Define expected patterns and find actual files
def find_file(pattern):
    files = glob.glob(pattern)
    if files:
        return files[0]
    return None

# 1. Identify Files
file_thesis_A = find_file('*ThesisData_A_OH_plus_others_MDS_Targets*.csv')
file_thesis_B = find_file('*ThesisData_B_FP_plus_others_MDS_Targets*.csv')

file_labels_A = find_file('*cluster_labels_matrix_samples_A_OH_plus_others*.csv')
file_labels_B = find_file('*cluster_labels_matrix_samples_B_FP_plus_others*.csv')

# Detail files (Uploaded in turn 27)
file_detail_OH_k50 = 'Detail_OH_Clusters_linear_top3_k50.csv'
file_detail_OH_k10 = 'Detail_OH_Clusters_linear_top3_k10.csv'
file_detail_FP_k50 = 'Detail_FP_Clusters_linear_top3_k50.csv'
file_detail_FP_k10 = 'Detail_FP_Clusters_linear_top3_k10.csv'

print(f"Found Thesis A: {file_thesis_A}")
print(f"Found Thesis B: {file_thesis_B}")
print(f"Found Labels A: {file_labels_A}")
print(f"Found Labels B: {file_labels_B}")

# Output files
OUT_TABLE_MAIN  = 'Table_4_x_Sweet_Spot_Summary.csv'
OUT_TABLE_OH_ALL = 'Table_Appendix_OH_All_Clusters_k50.csv'
OUT_TABLE_FP_ALL = 'Table_Appendix_FP_All_Clusters_k50.csv'

# Helper to load and merge
def load_and_merge(f_thesis, f_labels):
    if not f_thesis or not f_labels or not os.path.exists(f_thesis) or not os.path.exists(f_labels):
        print(f"[WARN] Missing files for merge: {f_thesis}, {f_labels}")
        return None
    
    try:
        df_t = pd.read_csv(f_thesis)
        if 'Unnamed: 0' in df_t.columns:
            df_t.rename(columns={'Unnamed: 0': 'ID'}, inplace=True)
        
        df_l = pd.read_csv(f_labels)
        # Merge on ID
        df = pd.merge(df_t, df_l, on='ID', how='inner')
        return df
    except Exception as e:
        print(f"[ERROR] Merge failed: {e}")
        return None

def load_detail(path):
    if not os.path.exists(path):
        print(f"[WARN] Detail file not found: {path}")
        return None
    return pd.read_csv(path)

# Helper to get stats
def get_stats(df, col_k, cluster_id, detail_df, type_name):
    subset = df[df[col_k] == cluster_id]
    if subset.empty:
        return {'Count': 0, 'Mean_PCE': 0, 'Std_Dev': 0, 'Max_PCE': 0, 'Major_Component': "N/A"}
        
    pce_data = subset['PCEmax']
    
    desc = "N/A"
    if detail_df is not None:
        row = detail_df[detail_df['Cluster_ID'] == cluster_id]
        if not row.empty:
            col_comp = 'Major_Material' if type_name == 'OH' else 'Major_Fragment'
            raw_name = str(row[col_comp].values[0])
            desc = raw_name.replace("SMILESsname", "").replace("p1M_", "").replace("nM_", "").replace("inM_", "")
    
    return {
        'Count': len(subset),
        'Mean_PCE': pce_data.mean(),
        'Std_Dev': pce_data.std(),
        'Max_PCE': pce_data.max(),
        'Major_Component': desc
    }

# Main Process
df_A = load_and_merge(file_thesis_A, file_labels_A)
df_B = load_and_merge(file_thesis_B, file_labels_B)

det_OH_k50 = load_detail(file_detail_OH_k50)
det_OH_k10 = load_detail(file_detail_OH_k10)
det_FP_k50 = load_detail(file_detail_FP_k50)
det_FP_k10 = load_detail(file_detail_FP_k10)

# --- Generate Table 4.x ---
if df_A is not None and df_B is not None:
    print("\nGenerating Table 4.x...")
    rows = []
    
    # 1. OH k=10 Best
    best_id = df_A.groupby('linear_top3_k10')['PCEmax'].mean().idxmax()
    st = get_stats(df_A, 'linear_top3_k10', best_id, det_OH_k10, 'OH')
    rows.append(['OH (Material)', 10, best_id, st['Count'], st['Mean_PCE'], st['Std_Dev'], st['Max_PCE'], st['Major_Component']])
    
    # 2. OH k=50 Best (Cluster 19)
    best_id_50 = 19
    st = get_stats(df_A, 'linear_top3_k50', best_id_50, det_OH_k50, 'OH')
    rows.append(['OH (Material)', 50, best_id_50, st['Count'], st['Mean_PCE'], st['Std_Dev'], st['Max_PCE'], st['Major_Component']])
    
    # 3. FP k=10 Best
    best_id = df_B.groupby('linear_top3_k10')['PCEmax'].mean().idxmax()
    st = get_stats(df_B, 'linear_top3_k10', best_id, det_FP_k10, 'FP')
    rows.append(['FP (Structure)', 10, best_id, st['Count'], st['Mean_PCE'], st['Std_Dev'], st['Max_PCE'], st['Major_Component']])
    
    # 4. FP k=50 Best (Cluster 18)
    best_id_50_fp = 18
    st = get_stats(df_B, 'linear_top3_k50', best_id_50_fp, det_FP_k50, 'FP')
    rows.append(['FP (Structure)', 50, best_id_50_fp, st['Count'], st['Mean_PCE'], st['Std_Dev'], st['Max_PCE'], st['Major_Component']])
    
    df_table4x = pd.DataFrame(rows, columns=['Dataset', 'Resolution (k)', 'Cluster ID', 'N', 'Mean PCE (%)', 'Std Dev', 'Max PCE (%)', 'Major Component'])
    df_table4x.to_csv(OUT_TABLE_MAIN, index=False)
    print(df_table4x)
    
    # --- Generate Appendix Tables (Full Ranking) ---
    
    # OH Ranking
    if det_OH_k50 is not None:
        cids = sorted(df_A['linear_top3_k50'].unique())
        rr = []
        for cid in cids:
            st = get_stats(df_A, 'linear_top3_k50', cid, det_OH_k50, 'OH')
            rr.append([cid, st['Count'], st['Mean_PCE'], st['Std_Dev'], st['Max_PCE'], st['Major_Component']])
        df_r_oh = pd.DataFrame(rr, columns=['Cluster ID', 'N', 'Mean PCE (%)', 'Std Dev', 'Max PCE (%)', 'Major Component'])
        # Fix sort column name (using correct space)
        df_r_oh = df_r_oh.sort_values('Mean PCE (%)', ascending=False)
        df_r_oh.to_csv(OUT_TABLE_OH_ALL, index=False)
        print(f"\nSaved OH Ranking to {OUT_TABLE_OH_ALL}")
        
    # FP Ranking
    if det_FP_k50 is not None:
        cids = sorted(df_B['linear_top3_k50'].unique())
        rr = []
        for cid in cids:
            st = get_stats(df_B, 'linear_top3_k50', cid, det_FP_k50, 'FP')
            rr.append([cid, st['Count'], st['Mean_PCE'], st['Std_Dev'], st['Max_PCE'], st['Major_Component']])
        df_r_fp = pd.DataFrame(rr, columns=['Cluster ID', 'N', 'Mean PCE (%)', 'Std Dev', 'Max PCE (%)', 'Major Component'])
        # Fix sort column name (using correct space)
        df_r_fp = df_r_fp.sort_values('Mean PCE (%)', ascending=False)
        df_r_fp.to_csv(OUT_TABLE_FP_ALL, index=False)
        print(f"Saved FP Ranking to {OUT_TABLE_FP_ALL}")

else:
    print("[ERROR] Could not load main dataframes.")

Found Thesis A: None
Found Thesis B: None
Found Labels A: None
Found Labels B: None
[WARN] Missing files for merge: None, None
[WARN] Missing files for merge: None, None
[WARN] Detail file not found: Detail_OH_Clusters_linear_top3_k50.csv
[WARN] Detail file not found: Detail_OH_Clusters_linear_top3_k10.csv
[WARN] Detail file not found: Detail_FP_Clusters_linear_top3_k50.csv
[WARN] Detail file not found: Detail_FP_Clusters_linear_top3_k10.csv
[ERROR] Could not load main dataframes.
